<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/project-2-llm-evaluation-suite/06b_regression_alarm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 6b: Weekly Regression Alarm and Production-Sampled Trace Dataset

**Goal:** Build a regression alarm that compares current evaluation scores against a prior baseline and flags meaningful drops, plus a production-sampling mechanism for selecting traces for human annotation. This is the last piece before Phase 7's synthesis.

**Design decision:** the alarm logic itself (threshold comparison, drift detection) and the sampling logic are both deterministic. Neither needs a model call, so both are built as real, working code from the start, no `SIMULATED_OUTPUT` branch needed for the core logic itself, the same approach used in Phase 5c's ATLAS mapping.

**Honest limitation, stated upfront:** a regression alarm's entire purpose is comparing *this week* against *prior weeks*. Project 2 has only ever run once, there is no real second data point yet. This notebook uses Phase 6a's saved scores as the one real baseline that exists, and constructs one clearly-labeled demonstration comparison point to prove the alarm logic actually fires when it should and stays quiet when it shouldn't. That demonstration point is not invented history presented as real, it is stated plainly as a synthetic test case for the alarm's own logic.

**Tools:** Langfuse v4, pandas

**Date:** July 2026

**Status:** In progress. Alarm and sampling logic are real. Real week-over-week comparison requires this suite to actually run more than once over time, which has not happened yet.

In [1]:
# Cell 2: Mount Drive and load Phase 6a as the real baseline

from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/"

phase6a_path = DRIVE_PATH + "phase06a_langfuse_custom_scores_results.json"
if os.path.exists(phase6a_path):
    with open(phase6a_path) as f:
        phase6a = json.load(f)
    print("Phase 6a results confirmed.")
    print(f"  Sources wired: {phase6a['trace_count']}")
    print(f"  Real: 0 | Simulated/Mixed: {phase6a['trace_count']} "
          f"(per Phase 6a's own findings)")
else:
    print("WARNING: Phase 6a results not found.")
    print(f"Expected: {phase6a_path}")
    print("Run 06a_langfuse_custom_scores.ipynb first.")

Mounted at /content/drive
Phase 6a results confirmed.
  Sources wired: 9
  Real: 0 | Simulated/Mixed: 9 (per Phase 6a's own findings)


In [2]:
# Cell 3: Install packages
# Same as Phase 5c: the alarm and sampling logic are deterministic and need
# no LLM API at all. Langfuse kept for consistency with every phase's trace
# logging convention.

!pip install langfuse pandas --quiet

print("Packages installed.")
print("No Gemini or Claude client needed in this notebook's core logic.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 11.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
Packages installed.
No Gemini or Claude client needed in this notebook's core logic.


In [3]:
# Cell 4: Simulated output flag and Langfuse client
# Same decoupling as Phase 6a: this flag only governs whether Langfuse
# traces are sent live. The alarm logic and sampling logic below are real,
# deterministic code regardless of this flag's value.

SIMULATED_OUTPUT = True

from google.colab import userdata

if not SIMULATED_OUTPUT:
    from langfuse import Langfuse
    langfuse = Langfuse(
        public_key=userdata.get('LANGFUSE_PUBLIC_KEY'),
        secret_key=userdata.get('LANGFUSE_SECRET_KEY'),
        host="https://cloud.langfuse.com"
    )
    print("Langfuse client initialised.")
else:
    print("[SIMULATED] Langfuse client not initialised (trace logging only).")
    print(f"SIMULATED_OUTPUT = {SIMULATED_OUTPUT}")

print()
print("Reminder: the regression alarm and sampling logic below are real,")
print("deterministic code. This flag affects only whether alarm-trigger")
print("events get logged to a live Langfuse dashboard.")

[SIMULATED] Langfuse client not initialised (trace logging only).
SIMULATED_OUTPUT = True

Reminder: the regression alarm and sampling logic below are real,
deterministic code. This flag affects only whether alarm-trigger
events get logged to a live Langfuse dashboard.


In [4]:
# Cell 5: Regression alarm logic
# Real, deterministic comparison logic, no model call involved. Compares a
# "current" set of scores against a "baseline" set and flags any metric
# that dropped more than the configured threshold.

REGRESSION_THRESHOLD = 0.05  # a 5-percentage-point drop triggers an alarm

def check_regression(current_scores: dict, baseline_scores: dict,
                      threshold: float = REGRESSION_THRESHOLD) -> list:
    """Compares current vs baseline headline scores per phase.
    Returns a list of alarm events for any metric that dropped by more
    than `threshold`. This function is real and permanent, not a
    placeholder, it will run identically on real data once real data
    exists."""
    alarms = []
    for phase_key in baseline_scores:
        if phase_key not in current_scores:
            continue
        baseline_val = baseline_scores[phase_key]["headline_score"]
        current_val = current_scores[phase_key]["headline_score"]
        drop = baseline_val - current_val
        if drop > threshold:
            alarms.append({
                "phase": phase_key,
                "metric": baseline_scores[phase_key]["headline_label"],
                "baseline_score": baseline_val,
                "current_score": current_val,
                "drop": round(drop, 4),
                "severity": "ALARM" if drop > (threshold * 2) else "WARNING",
            })
    return alarms


# The one real baseline that exists: Phase 6a's normalized scores.
baseline_scores = phase6a["normalized_scores"]

print(f"Regression alarm logic loaded. Threshold: {REGRESSION_THRESHOLD} "
      f"({REGRESSION_THRESHOLD:.0%} drop triggers a warning, "
      f"{REGRESSION_THRESHOLD*2:.0%} triggers an alarm).")
print(f"Baseline loaded from Phase 6a: {len(baseline_scores)} phases.")

Regression alarm logic loaded. Threshold: 0.05 (5% drop triggers a warning, 10% triggers an alarm).
Baseline loaded from Phase 6a: 9 phases.
